## <font color='turquoise'>**Reading the model and necessary packages:**</font>

To read the model, one must first define the file path. The model’s file path is stored in `iPrub22_path` using the `pathlib` function from the `Path` library. To read the model’s content, the `read_sbml_model` function from `cobra.io` is utilized. Subsequently, iPrub22_Reconstruction can be used to easily access the model’s content.

Reminder:
Ensure that the `home_path` variable is correctly set. It should be equal to the path of the directory where your model file is located.

In [3]:
import cobra
import cobra.util
from cobra.io import read_sbml_model
from pathlib import Path

# Path to the SBML file
iPrub22_path = "../../iPrub22_Reconstruction.sbml"

# Read SBML model
iPrub22 = cobra.io.read_sbml_model(iPrub22_path)

Set parameter Username
Set parameter LicenseID to value 2682297
Academic license - for non-commercial use only - expires 2026-06-25


<font color='blue'>**Minimum constraints for biomass production:**</font>

- Uptake_015: Alpha-glucose (Carbon source)
- Uptake_062: FE2+
- Uptake_130: NH3 (Nitrogen source)
- Uptake_136: Oxygen-molecule or Uptake_077: Hydrogen-peroxide 
- Uptake_146: Phosphate
- Uptake_157: Riboflavin or Uptake_065: FMN 
- Uptake_169: Elemental-Sulfur
- Uptake_171: Thiamine

<font color='blue'>**Constraints for specialised metabolites production:**</font>

**Production of Isoepoxydon, Roquefortins and assimilates, Averufin, Versicolorin B and Versiconol**
- Uptake_155: Red-NADPH-Hemoprotein-Reductases

**Production of Penicillin K**
- Uptake_044: Decanoate or Uptake_137: Octanoate

**Production of Peniciilin V**
- Uptake_144: Phenoxyacetate

**Production of Sterigmatocystin and Aflatoxin B1**
- Uptake_008: 6-demethylsterigmatocystin            
- Uptake_155: Red-NADPH-Hemoprotein-Reductases

## <font color='turquoise'>**Default model:**</font>

### <font color='magenta'>**Check objective function and identify the constraints:**</font>

In [4]:
# Optimize the model
solution = iPrub22.optimize()

# Print the objective value
print(solution.objective_value)

0.294447219323639


In [5]:
iPrub22.summary()

Metabolite,Reaction,Flux,C-Number,C-Flux
ALPHA-GLUCOSE_e,Uptake_015,15,6,100.00%
FE+2_e,Uptake_062,5.889E-05,0,0.00%
AMMONIA_e,Uptake_130,2.062,0,0.00%
OXYGEN-MOLECULE_e,Uptake_136,19.76,0,0.00%
Pi_e,Uptake_146,0.09683,0,0.00%
RIBOFLAVIN_e,Uptake_157,2.944E-05,17,0.00%
Elemental-Sulfur_e,Uptake_169,0.07947,0,0.00%
THIAMINE_e,Uptake_171,2.944E-05,12,0.00%
Metabolite,Reaction,Flux,C-Number,C-Flux
Biomass_e,Exchange_Biomass,-0.2944,0,0.00%


In [6]:
def print_constraints(model):
    print("MinConstraints:")
    for reaction in model.reactions:
        if reaction.lower_bound > -1000 and reaction.lower_bound != 0 :
            print(f"{reaction.id}\t{round(reaction.lower_bound)}")

    print("\nMaxConstraints:")
    for reaction in model.reactions:
        if reaction.upper_bound < 1000 and reaction.upper_bound != 0:  
            print(f"{reaction.id}\t{round(reaction.upper_bound)}")

print_constraints(iPrub22)

MinConstraints:
NGAM	1

MaxConstraints:
Uptake_008	10
Uptake_015	15
Uptake_044	10
Uptake_062	10
Uptake_130	5
Uptake_144	10
Uptake_146	10
Uptake_155	10
Uptake_157	10
Uptake_169	10
Uptake_171	10
NGAM	1


### <font color='magenta'>**Flux Balance Analysis result:**</font>

In [7]:
iPrub22.reactions.Biomass_rxn.summary()

In [8]:
iPrub22.metabolites.Biomass_c.summary()

Percent,Flux,Reaction,Definition
100.00%,0.2944,Biomass_rxn,0.04 AAPOOL_c + 104.0 ATP_c + 0.25 CELLWALL_c + 0.0001 COF_c + 0.01 DNA_c + 0.035 PLIPIDS_c + 0.45 PROTEIN_c + 0.08 RNA_c + 104.0 WATER_c --> 104.0 ADP_c + Biomass_c + 104.0 Pi_c
Percent,Flux,Reaction,Definition
100.00%,-0.2944,Transport_Biomass,Biomass_c --> Biomass_e


### <font color='magenta'>**List of monitored metabolites:**</font>

We create the `get_monitored_meta` function to retrieve specific information about certain metabolites in the iPrub22_model. 

The function retrieves the following information for each metabolite:
- Name
- Formula
- Charge
- ID
- Exchange reaction
- Equations for transport and production

The function takes two lists as inputs: `list_of_metabolites` (a list of metabolites to search for) and `list_of_exchange_rxn` (a manually created list of exchange reactions for these metabolites).

For each metabolite in `list_of_metabolites`, the function searches the model for a matching metabolite. If a match is found, it retrieves and prints the requested information. The enumerate function is used to get the index of each metabolite in the list, which is then used to access the corresponding exchange reaction from `list_of_exchange_rxn`.

Each metabolite has two associated reactions: transport (which moves the metabolite from the cytosol to the extracellular space) and production (which represents the exit of the metabolite from the system). These are dead-end reactions and are initially blocked in the model. The function unblocks these reactions for the metabolites of interest.

All the reactions of transports are balanced and all the reaction of production are in balanced

The function does return a list for the readble names of metabolites (`metabolites_readble_names`) and it prints the retrieved information. It could be modified to return a data structure containing the information if needed. If a metabolite in `list_of_metabolites` is not found in the model, the function simply skips it (you can replace this with the actual behavior if different).

In [16]:
# Attention nom du modèle écrit en dur dans la fonction - à passer en paramètre
def get_monitored_meta(list_of_metabolites, list_of_exchange_rxn):
    metabolites_readble_names = []
    for i, met_id in enumerate(list_of_metabolites):
        for metabolite in iPrub22.metabolites:
            if metabolite.id == met_id:
                print("\033[1m" + "Metabolite Name : " + "\033[0m", metabolite.name)
                metabolites_readble_names.append(metabolite.name)
                print("\033[1m" + "Formula : " + "\033[0m", metabolite.formula)
                print("\033[1m" + "Charge : " + "\033[0m", metabolite.charge)
                print("\033[1m" + "Metabolite ID : " + "\033[0m", metabolite.id)
                print("\033[1m" + "Exchange reaction : " + "\033[0m", list_of_exchange_rxn[i])
                for reaction in iPrub22.reactions:
                    if 'Transport' in reaction.name:  
                        if met_id in [met.id for met in reaction.metabolites]:
                            print("\033[1m" + "Equation Transport: " + "\033[0m", reaction.build_reaction_string(use_metabolite_names=False))
                            reaction_exit = iPrub22.reactions.get_by_id(list_of_exchange_rxn[i])
                            print("\033[1m" + "Equation production: " + "\033[0m", reaction_exit.build_reaction_string(use_metabolite_names=False))
                            print('\n')
    return metabolites_readble_names

#### <font color='green'>**Penicillins Biosynthesis**</font>

In [17]:
metabolites_1 = ['6-AMINOPENICILLANATE_c', 'CPD-1772_c', 'ISOPENICILLIN-N_c', 'CPD-9122_c', 'CPD-9196_c', 'PENICILLIN-N_c']
rxn_echange_1 = ['Production_001', 'Production_004', 'Production_010', 'Production_012', 'Production_013', 'Production_015']
names_1 = []
names_1 = get_monitored_meta(metabolites_1, rxn_echange_1)

Metabolite Name :  6-aminopenicillanate
Formula :  C8H12N2O3S
Charge :  0
Metabolite ID :  6-AMINOPENICILLANATE_c
Exchange reaction :  Production_001
Equation Transport:  6-AMINOPENICILLANATE_c <=> 6-AMINOPENICILLANATE_e
Equation production:  6-AMINOPENICILLANATE_e --> 


Metabolite Name :  2-aminoacetaldehyde
Formula :  C2H6NO
Charge :  1
Metabolite ID :  CPD-1772_c
Exchange reaction :  Production_004
Equation Transport:  CPD-1772_c <=> CPD-1772_e
Equation production:  PENICILLIN-G_e --> 


Metabolite Name :  isopenicillin N
Formula :  C14H20N3O6S
Charge :  -1
Metabolite ID :  ISOPENICILLIN-N_c
Exchange reaction :  Production_010
Equation Transport:  ISOPENICILLIN-N_c <=> ISOPENICILLIN-N_e
Equation production:  ISOPENICILLIN-N_e --> 


Metabolite Name :  penicillin K
Formula :  C16H26N2O4S
Charge :  0
Metabolite ID :  CPD-9122_c
Exchange reaction :  Production_012
Equation Transport:  CPD-9122_c <=> CPD-9122_e
Equation production:  CPD-9122_e --> 


Metabolite Name :  penicillin V
For

#### <font color='green'>**Roquefortines Biosynthesis**</font>

In [18]:
metabolites_2 = ['CPD-17378_c', 'CPD-17379_c', 'CPD-17380_c', 'CPD-17381_c', 'CPD-17389_c', 'CPD-17392_c', 'CPD-17393_c', 'CPD-17394_c', 'CPD-17395_c', 'CPD-17390_c', 'CPD-17391_c']
rxn_echange_2 = ['Production_022', 'Production_023', 'Production_024', 'Production_025', 'Production_026', 'Production_027', 'Production_028', 'Production_029', 'Production_030', 'Production_031', 'Production_032']
names_2 = []
names_2 = get_monitored_meta(metabolites_2, rxn_echange_2)

Metabolite Name :  histidyltryptophyldiketopiperazine
Formula :  C17H17N5O2
Charge :  0
Metabolite ID :  CPD-17378_c
Exchange reaction :  Production_022
Equation Transport:  CPD-17378_c --> CPD-17378_e
Equation production:  CPD-17378_e --> 


Metabolite Name :  dehydrohistidyltryptophyldiketopiperazine
Formula :  C17H15N5O2
Charge :  0
Metabolite ID :  CPD-17379_c
Exchange reaction :  Production_023
Equation Transport:  CPD-17379_c --> CPD-17379_e
Equation production:  CPD-17379_e --> 


Metabolite Name :  roquefortine D
Formula :  C22H25N5O2
Charge :  0
Metabolite ID :  CPD-17380_c
Exchange reaction :  Production_024
Equation Transport:  CPD-17380_c --> CPD-17380_e
Equation production:  CPD-17380_e --> 


Metabolite Name :  roquefortine C
Formula :  C22H23N5O2
Charge :  0
Metabolite ID :  CPD-17381_c
Exchange reaction :  Production_025
Equation Transport:  CPD-17381_c --> CPD-17381_e
Equation production:  CPD-17381_e --> 


Metabolite Name :  glandicoline A
Formula :  C22H21N5O3
Charg

#### <font color='green'>**Isoepoxydon biosynthesis and toluquinol precursor**</font>

In [22]:
metabolites_3 = ['CPD-112_c', 'CPD-16723_c', 'CPD-16720_c']
rxn_echange_3 = ['Production_016', 'Production_020', 'Production_021']
names_3 = []
names_3 = get_monitored_meta(metabolites_3, rxn_echange_3)

Metabolite Name :  3-methylphenol
Formula :  C7H8O
Charge :  0
Metabolite ID :  CPD-112_c
Exchange reaction :  Production_016
Equation Transport:  CPD-112_c --> CPD-112_e
Equation production:  CPD-112_e --> 


Metabolite Name :  isoepoxydon
Formula :  C7H8O4
Charge :  0
Metabolite ID :  CPD-16723_c
Exchange reaction :  Production_020
Equation Transport:  CPD-16723_c --> CPD-16723_e
Equation production:  CPD-16723_e --> 


Metabolite Name :  toluquinol
Formula :  C7H8O2
Charge :  0
Metabolite ID :  CPD-16720_c
Exchange reaction :  Production_021
Equation Transport:  CPD-16720_c --> CPD-16720_e
Equation production:  CPD-16720_e --> 




#### <font color='green'>**Precursor of DHN-melanin**</font>

In [23]:
metabolites_4 = ['CPD-18255_c']
rxn_echange_4 = ['Production_033']
names_4 = []
names_4 = get_monitored_meta(metabolites_4, rxn_echange_4)

Metabolite Name :  YWA1
Formula :  C14H12O6
Charge :  0
Metabolite ID :  CPD-18255_c
Exchange reaction :  Production_033
Equation Transport:  CPD-18255_c --> CPD-18255_e
Equation production:  CPD-18255_e --> 




#### <font color='green'>**Ferrichrome biosynthesis**</font>

In [24]:
metabolites_5 = ['CPD0-2241_c', 'CPD-17185_c']
rxn_echange_5 = ['Production_034', 'Production_035']
names_5 = []
names_5 = get_monitored_meta(metabolites_5, rxn_echange_5)

Metabolite Name :  ferrichrome
Formula :  C27H42FeN9O12
Charge :  0
Metabolite ID :  CPD0-2241_c
Exchange reaction :  Production_034
Equation Transport:  CPD0-2241_c --> CPD0-2241_e
Equation production:  CPD0-2241_e --> 


Metabolite Name :  ferrichrome A
Formula :  C41H58FeN9O20
Charge :  0
Metabolite ID :  CPD-17185_c
Exchange reaction :  Production_035
Equation Transport:  CPD-17185_c --> CPD-17185_e
Equation production:  CPD-17185_e --> 




#### <font color='green'>**Precursor of Pr-Toxin**</font>

In [25]:
metabolites_6 = ['ARISTOLOCHENE_c']
rxn_echange_6 = ['Production_036']
names_6 = []
names_6 = get_monitored_meta(metabolites_6, rxn_echange_6)

Metabolite Name :  aristolochene
Formula :  C15H24
Charge :  0
Metabolite ID :  ARISTOLOCHENE_c
Exchange reaction :  Production_036
Equation Transport:  ARISTOLOCHENE_c --> ARISTOLOCHENE_e
Equation production:  ARISTOLOCHENE_e --> 




#### <font color='green'>**Precursor of Andrastin A**</font>

In [26]:
metabolites_7 = ['CPD-17316_c']
rxn_echange_7 = ['Production_037']
names_7 = []
names_7 = get_monitored_meta(metabolites_7, rxn_echange_7)

Metabolite Name :  3,5-dimethylorsellinate
Formula :  C10H11O4
Charge :  -1
Metabolite ID :  CPD-17316_c
Exchange reaction :  Production_037
Equation Transport:  CPD-17316_c --> CPD-17316_e
Equation production:  CPD-17316_e --> 




#### <font color='green'>**Biosynthesis of Averufin/Versiconol/Versicolorin B/Sterigmatocystin/Aflatoxin B1**</font>

In [27]:
metabolites_8 = ['CPD-10167_c', 'CPD-10175_c', 'CPD-10174_c', 'STERIGMATOCYSTIN_c', 'CPD-4592_c']
rxn_echange_8 = ['Production_038', 'Production_039', 'Production_040', 'Production_041', 'Production_042']
names_8 = []
names_8 = get_monitored_meta(metabolites_8, rxn_echange_8)

Metabolite Name :  (1'S,5'S)-averufin
Formula :  C20H16O7
Charge :  0
Metabolite ID :  CPD-10167_c
Exchange reaction :  Production_038
Equation Transport:  CPD-10167_c --> CPD-10167_e
Equation production:  CPD-10167_e --> 


Metabolite Name :  versicolorin B
Formula :  C18H11O7
Charge :  -1
Metabolite ID :  CPD-10175_c
Exchange reaction :  Production_039
Equation Transport:  CPD-10175_c --> CPD-10175_e
Equation production:  CPD-10175_e --> 


Metabolite Name :  versiconol
Formula :  C18H16O8
Charge :  0
Metabolite ID :  CPD-10174_c
Exchange reaction :  Production_040
Equation Transport:  CPD-10174_c --> CPD-10174_e
Equation production:  CPD-10174_e --> 


Metabolite Name :  sterigmatocystin
Formula :  C18H12O6
Charge :  0
Metabolite ID :  STERIGMATOCYSTIN_c
Exchange reaction :  Production_041
Equation Transport:  STERIGMATOCYSTIN_c --> STERIGMATOCYSTIN_e
Equation production:  STERIGMATOCYSTIN_e --> 


Metabolite Name :  aflatoxin B1
Formula :  C17H12O6
Charge :  0
Metabolite ID :  CPD-

### <font color='magenta'>**Flux Variance Analysis (FVA):**</font>

In [29]:
from cobra.flux_analysis import flux_variability_analysis

Production_specialised_metabolites = ['Production_001', 'Production_004', 'Production_010', 'Production_012', 'Production_013', 'Production_015', 'Production_022', 'Production_023', 'Production_024', 'Production_025', 'Production_026', 'Production_027', 'Production_028', 'Production_029', 'Production_030', 'Production_031', 'Production_032', 'Production_016', 'Production_020', 'Production_021', 'Production_033', 'Production_034', 'Production_035', 'Production_036', 'Production_037', 'Production_038', 'Production_039', 'Production_040', 'Production_041', 'Production_042']

Variation = flux_variability_analysis(iPrub22, Production_specialised_metabolites, fraction_of_optimum = 0.8)

# Concatanation of name lists
List_of_Readble_Names = names_1 + names_2 + names_3 + names_4 + names_5 + names_6 + names_7 + names_8

In [31]:
import pandas as pd

df_names = pd.DataFrame(List_of_Readble_Names, columns=['Production of Metabolites'])

# Reset the index of your Variation dataframe
Variation_reset = Variation.reset_index()
Variation_reset = round(Variation_reset, 3)

# Concatenate the two dataframes
result = pd.concat([df_names, Variation_reset], axis=1)

# Remove the index
result_no_index = result.set_index('Production of Metabolites')

# Remove the 'index' column
result_final = result_no_index.drop(columns=['index'])

print(result_final)

                                           minimum  maximum
Production of Metabolites                                  
6-aminopenicillanate                           0.0    1.675
2-aminoacetaldehyde                            0.0    1.167
isopenicillin N                                0.0    1.117
penicillin K                                   0.0    1.675
penicillin V                                   0.0    1.675
penicillin N                                   0.0    1.117
histidyltryptophyldiketopiperazine             0.0    0.670
dehydrohistidyltryptophyldiketopiperazine      0.0    0.670
roquefortine D                                 0.0    0.670
roquefortine C                                 0.0    0.670
glandicoline A                                 0.0    0.670
N1-hydroxy-roquefortine C                      0.0    0.670
roquefortine F                                 0.0    0.670
neoxaline                                      0.0    0.670
roquefortine L                          